# 第17周 · DDPM 是 Flow Matching 的一个特例；条件生成与 classifier-free guidance

**本周结束时你能做到**：说出 DDPM 的前向加噪就是"另一条高斯插值"；在第 16 周的代码上**只改两行**得到 DDPM 的训练，再写出它的祖先采样；用工具包的换算函数在两套**相反的时间约定**之间来回换算，理解 ε、数据、速度三种预测可以互相换算；理解随机采样与确定性 ODE 采样的区别；给网络加上类别嵌入，实现 classifier-free guidance，并诚实地评估"判为目标数字的比例"与多样性之间的权衡；知道"在潜空间中生成"（Stable Diffusion）的思路。

本周的蓝本：

- **DDPM 原文**：Ho, Jain & Abbeel, *Denoising Diffusion Probabilistic Models*, NeurIPS 2020（[arXiv:2006.11239](https://arxiv.org/abs/2006.11239)）——前向过程 eq. (4)、线性 β 调度与 T = 1000（§4）、训练 Algorithm 1、采样 Algorithm 2。"论文 → 代码"的逐行对照阅读：Rogge & Rasul, [The Annotated Diffusion Model](https://huggingface.co/blog/annotated-diffusion)（Hugging Face 博客，2022；配套 notebook 为 Apache-2.0）。
- **MIT 6.S184 讲义**（Holderrieth & Erives, [arXiv:2506.02070](https://arxiv.org/abs/2506.02070)）：§4.3 Example 23 把 DDPM 写成同一框架下"换一个预测目标"；附录 E Summary 45 列出文献中相反的时间约定。
- **ε / 数据 / v 的换算**：v-prediction 出自 Salimans & Ho, *Progressive Distillation for Fast Sampling of Diffusion Models*, ICLR 2022（[arXiv:2202.00512](https://arxiv.org/abs/2202.00512)）§4。
- **Classifier-free guidance**：Ho & Salimans, *Classifier-Free Diffusion Guidance*（[arXiv:2207.12598](https://arxiv.org/abs/2207.12598)）Algorithm 1–2 与 eq. (6)。条件生成的另一份可读实现：[TeaPearce/Conditional_Diffusion_MNIST](https://github.com/TeaPearce/Conditional_Diffusion_MNIST)（MIT License，只作阅读参考，见 S3 的说明）。

> **本周的约定提醒**：第 16 周我们把噪声叫 $x_0$、数据叫 $x_1$；DDPM 论文里 $x_0$ 却是**数据**。为了不混淆，本周代码一律写 `x_data` 和 `eps`。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision.utils import make_grid
from zero2researcher import data, experiments, generative
from zero2researcher.checks import week17 as ck

# 2D 部分固定用 CPU；MNIST 部分自动选择：Colab 上是 CUDA，Apple 芯片是 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| MNIST 部分的 device:", device)

In [ ]:
# ---------- 第 16 周写过的零件（原样给出） ----------
class VelocityMLP(nn.Module):
    """输入 (x, t) 的 3→128→128→2 MLP。FM 里它输出速度；本周 DDPM 用同一个类，输出的含义变成噪声 ε"""

    def __init__(self, hidden=128, depth=2):
        super().__init__()
        layers, d = [], 3
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.SiLU()]
            d = hidden
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        t = torch.as_tensor(t, dtype=x.dtype).reshape(-1, 1).expand(len(x), 1)
        return self.net(torch.cat([x, t], dim=1))


def euler_integrate(u, x0, n_steps, t_start=0.0, t_end=1.0):
    h = (t_end - t_start) / n_steps
    x, path = x0, [x0]
    for i in range(n_steps):
        x = x + h * u(x, t_start + i * h)
        path.append(x)
    return torch.stack(path)


@torch.no_grad()
def fm_sample(model, x_init, n_steps):
    # FM 采样：从给定的初始噪声出发，欧拉法从 t = 0（噪声）走到 t = 1（数据），只返回终点
    return euler_integrate(model, x_init, n_steps)[-1]


def noise(shape, seed=0):
    # 在 CPU 上用固定种子抽标准正态噪声：换设备也得到同一份噪声
    return torch.randn(shape, generator=torch.Generator().manual_seed(seed))


def show_grid(images, nrow=8, title="", ax=None):
    grid = make_grid(images.detach().cpu().clamp(-1, 1), nrow=nrow, normalize=True, value_range=(-1, 1))
    if ax is None:
        plt.figure(figsize=(4.5, 4.5))
        ax = plt.gca()
    ax.imshow(grid.permute(1, 2, 0))
    ax.axis("off")
    ax.set_title(title)

---
## S1 · DDPM 的前向过程：另一条高斯插值

DDPM 从数据出发，一步步往里**加噪声**，第 $t$ 步加的噪声方差是 $\beta_t$。原文 §4 取 $T = 1000$ 步、$\beta_t$ 从 $10^{-4}$ **线性**增到 $0.02$。记 $\alpha_t = 1 - \beta_t$、$\bar\alpha_t = \prod_{s \le t} \alpha_s$，原文 eq. (4) 给出任意一步的闭式：

$$x_t = \sqrt{\bar\alpha_t}\; x_{\text{data}} + \sqrt{1 - \bar\alpha_t}\; \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

**$t = 0$ 是数据**，$\bar\alpha$ 从 1 一路降到接近 0（几乎纯噪声）。

把它和第 16 周的 $x_t = t\,x_{\text{data}} + (1 - t)\,\varepsilon$ 并排看：**两者都是"数据 × 一个系数 + 噪声 × 另一个系数"**，只是系数随时间变化的方式不同（而且时间方向相反）。

为了让第 16 周的训练代码一个字都不用改（时间仍然是 `torch.rand` 抽的 $[0, 1]$ 之间的数），代码里用离散调度的**连续版**：令 $\tau = t / T \in [0, 1]$。因为 $\beta$ 很小，$\ln \bar\alpha_t = \sum \ln(1 - \beta_s) \approx -\sum \beta_s$——把 1000 个线性增长的小 $\beta$ 加起来，就是对 $\beta(s) = 0.1 + 19.9\,s$ 积分：

$$\bar\alpha(\tau) = \exp\!\big(-(0.1\,\tau + 9.95\,\tau^2)\big)$$

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T, dtype=torch.float64)       # 原文 §4：β 从 1e-4 线性增到 0.02
# 在这里写你的代码
raise NotImplementedError


def alpha_bar(tau):
    # 连续版：τ = t/T ∈ [0, 1]，τ = 0 是数据（ᾱ = 1）
    # 在这里写你的代码
    raise NotImplementedError


ck.alpha_bar(alpha_bar)
tau_grid = torch.arange(1, T + 1, dtype=torch.float64) / T
print(f"离散 ᾱ_T = {float(alpha_bars_discrete[-1]):.2e} | 连续 ᾱ(1) = {float(alpha_bar(torch.tensor(1.0))):.2e}")
print(f"两者在 τ = t/T 处的最大差: {float((alpha_bars_discrete - alpha_bar(tau_grid)).abs().max()):.1e}")

In [ ]:
grid = torch.linspace(0, 1, 201)
ab = alpha_bar(grid)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.9))
axes[0].plot(grid, grid, label="data coefficient t")
axes[0].plot(grid, 1 - grid, label="noise coefficient 1 − t")
axes[0].set_xlabel("FM time t   (t = 0: noise, t = 1: data)")
axes[0].set_title("Flow Matching: straight-line mix")
axes[1].plot(grid, ab.sqrt(), label="data coefficient √ᾱ(τ)")
axes[1].plot(grid, (1 - ab).sqrt(), label="noise coefficient √(1 − ᾱ(τ))")
axes[1].plot(tau_grid[::50], alpha_bars_discrete[::50].sqrt(), "k.", ms=4, label="√ᾱ_t, discrete T = 1000")
axes[1].set_xlabel("DDPM time τ = t/T   (τ = 0: data, τ = 1: noise)")
axes[1].set_title("DDPM: variance-preserving mix")
axes[2].plot(grid, generative.ddpm_to_fm_time(ab), label="FM time with the same noise level")
axes[2].plot(grid, 1 - grid, "--", color="gray", label="naive flip 1 − τ (wrong)")
axes[2].set_xlabel("DDPM time τ")
axes[2].set_ylabel("FM time t")
axes[2].set_title("Converting time is not just flipping it")
for ax in axes:
    ax.legend(fontsize=8)
fig.suptitle("Both paths blend data and noise; they differ in schedule and in the direction of time")
plt.tight_layout()
plt.show()

> ⚠️ **时间约定警示**。FM（MIT 讲义、Lipman、Rectified Flow）：**$t = 0$ 是噪声**；DDPM：**$t = 0$ 是数据**。MIT 讲义附录 E（Summary 45）把后者明确列为文献中的反转约定（原文："Inverted time convention: t=0 corresponds to p_data"）。还有第三套：Google DeepMind 的博客 [*Diffusion Meets Flow Matching*](https://diffusionflow.github.io/) 取 $t = 0$ 为数据、$x_t = (1-t)\,x + t\,\varepsilon$，把"速度"定义成 $\varepsilon - x$——和我们的 $x_{\text{data}} - \varepsilon$ **正好差一个负号**。读代码和论文时，先找到"$t = 0$ 是什么"和"速度指向哪边"。

右图说明了**换算不是简单的 $t \leftrightarrow 1 - \tau$**：两条路径的噪声随时间增长得不一样快。正确的对齐方式是让**信噪比**相同——数据系数与噪声系数之比的平方：FM 是 $t^2/(1-t)^2$，DDPM 是 $\bar\alpha/(1-\bar\alpha)$。信噪比相同的两个状态**只差一个缩放因子** $\sqrt{\bar\alpha} + \sqrt{1-\bar\alpha}$。工具包把这些换算写成了小函数（`generative.ddpm_to_fm_time`、`fm_to_ddpm_time`、`ddpm_to_fm`、`fm_to_ddpm`），我们验证一下：

In [ ]:
torch.manual_seed(0)
x_data = generative.sample_moons(2000)
eps = torch.randn_like(x_data)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for col, tau in enumerate((0.0, 0.15, 0.3, 0.5, 1.0)):
    ab_tau = alpha_bar(torch.tensor(tau))
    x_vp = ab_tau.sqrt() * x_data + (1 - ab_tau).sqrt() * eps        # DDPM 状态
    x_fm, t_fm = generative.ddpm_to_fm(x_vp, ab_tau)                    # 换到 FM 约定
    gap = (x_fm - (t_fm * x_data + (1 - t_fm) * eps)).abs().max().item()
    print(f"DDPM τ = {tau:.2f}（ᾱ = {float(ab_tau):.3f}）↔ FM t = {float(t_fm):.3f}；"
          f"换算后与同一对 (x_data, ε) 的 FM 插值相差 {gap:.1e}")
    for row, (pts, title) in enumerate(((x_vp, f"DDPM: tau = {tau:.2f}"), (x_fm, f"FM: t = {float(t_fm):.2f}"))):
        axes[row, col].scatter(pts[:, 0], pts[:, 1], s=1, color=f"C{row}")
        axes[row, col].set_title(title)
        axes[row, col].set_xlim(-6, 6)
        axes[row, col].set_ylim(-4, 4)
        axes[row, col].set_aspect("equal")
fig.suptitle("Matched noise levels: the two states are the same picture up to a zoom factor")
plt.tight_layout()
plt.show()

### 在第 16 周的代码上只改两行

DDPM 原文 Algorithm 1 的训练：抽数据、抽 $t$、抽噪声，按 eq. (4) 加噪，让网络**预测噪声 $\varepsilon$**，损失是 MSE。和第 16 周对照：

| | Flow Matching（第 16 周） | DDPM（本周） |
| --- | --- | --- |
| 概率路径 | `xt = t * x_data + (1 - t) * eps` | `ab = alpha_bar(t)`；`xt = ab.sqrt() * x_data + (1 - ab).sqrt() * eps` |
| 预测目标 | `target = x_data - eps`（速度） | `target = eps`（噪声） |
| 其余 | 同一个网络、同一个优化器、同一个 MSE 损失、同样抽 `t = torch.rand(B, 1)` | 完全相同 |

**这本身就是一次"受控修改"的示范**：只改你想研究的东西，其他一切保持不变——作业的研究实验 4 就建立在这一点上。

In [ ]:
def train_2d(sample_data, kind="fm", steps=3000, batch_size=512, seed=0):
    """同一个网络、同一个训练循环；kind 只决定两行：概率路径与预测目标"""
    experiments.set_seed(seed)
    model = VelocityMLP()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(steps):
        x_data = sample_data(batch_size)
        eps = torch.randn_like(x_data)
        t = torch.rand(batch_size, 1)       # FM 里它是 t（0 = 噪声）；DDPM 里同一个数是 τ（0 = 数据）
        # 在这里写你的代码
        raise NotImplementedError
        loss = ((model(xt, t) - target) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model


t0 = time.time()
ddpm_moons = train_2d(generative.sample_moons, "ddpm")
print(f"DDPM（月牙）3000 步用时 {time.time() - t0:.1f} 秒")

### 祖先采样：去掉一点噪声，再加回一点新的噪声

训练好的网络 $\varepsilon_\theta(x, \tau)$ 估计"$x$ 里混进了什么噪声"。DDPM 原文 Algorithm 2 从纯噪声 $x_T \sim \mathcal{N}(0, I)$ 出发，$t = T, \dots, 1$ 逐步往回走：

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar\alpha_t}}\;\varepsilon_\theta(x_t, t)\Big) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I)$$

括号里是"减掉估计出的一部分噪声"，最后一项是"再加回一点**新的**随机噪声"——所以 DDPM 的采样是**随机的**；第 16 周的欧拉法是**确定性的**（同一个起点永远走到同一个终点）。$\sigma_t^2$ 原文给了两种取法、效果相近，我们取 $\sigma_t^2 = \tilde\beta_t = \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\beta_t$。

想用 $n$ 步（而不是 1000 步）采样时，把 $\tau$ 从 1 到 0 等分成 $n$ 段，把相邻两个网格点当作"相邻的两步"：$\alpha := \bar\alpha(\tau_{\text{当前}})/\bar\alpha(\tau_{\text{下一个}})$，其余公式不变（$n = 1000$ 时就是原文的 Algorithm 2）。最后一步走到 $\tau = 0$ 时 $\bar\alpha = 1$，$\sigma$ 自动为 0。

In [ ]:
@torch.no_grad()
def ddpm_sample(eps_model, x_init, n_steps, seed=0):
    """DDPM 祖先采样（原文 Algorithm 2），在 τ 网格 1 → 0 上走 n_steps 步；eps_model(x, τ) 预测噪声"""
    g = torch.Generator().manual_seed(seed)
    x = x_init
    taus = torch.linspace(1, 0, n_steps + 1)
    for i in range(n_steps):
        ab, ab_next = alpha_bar(taus[i]), alpha_bar(taus[i + 1])
        alpha = ab / ab_next                 # 这一大步的 α（n_steps = 1000 时就是原文的 α_t）
        # 在这里写你的代码
        raise NotImplementedError
    return x


moons_ref = generative.sample_moons(2000, rng=2026)       # 固定的真实参考样本
xs = ddpm_sample(ddpm_moons, noise((2000, 2)), 100)
ed, prec = generative.energy_distance(xs, moons_ref), generative.knn_precision(xs, moons_ref)
print(f"DDPM 祖先采样 100 步：能量距离 {ed:.4f} | k-NN precision {prec:.3f}")
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(moons_ref[:, 0], moons_ref[:, 1], s=2, color="lightgray", label="Real")
ax.scatter(xs[:, 0], xs[:, 1], s=2, label="DDPM, 100 ancestral steps")
ax.set_xlim(-6, 6)
ax.set_ylim(-4, 4)
ax.set_aspect("equal")
ax.legend(markerscale=4, fontsize=8)
ax.set_title("Two changed training lines plus a new sampler give DDPM")
plt.show()

诚实的说法是：**训练改 2 行，采样换 1 个函数**。（也可以不写新采样器：用下一节的换算把 $\varepsilon$ 转成速度，继续用欧拉法——但我们会看到那样效果更差。）

---
## S2 · 同一个任务、同一个网络：FM vs DDPM

用**同一个种子**再训练一个 FM 模型（同一个网络类、同一个训练循环，只是 `kind="fm"`），然后在 $\{2, 5, 20, 100\}$ 步下比较两种方法的样本：

In [ ]:
fm_moons = train_2d(generative.sample_moons, "fm")
STEPS = (2, 5, 20, 100)
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for col, n in enumerate(STEPS):
    for row, (name, xs) in enumerate((("FM + Euler", fm_sample(fm_moons, noise((2000, 2)), n)),
                                      ("DDPM + ancestral", ddpm_sample(ddpm_moons, noise((2000, 2)), n)))):
        ed, prec = generative.energy_distance(xs, moons_ref), generative.knn_precision(xs, moons_ref)
        print(f"{name:17s} {n:3d} 步：能量距离 {ed:.4f} | precision {prec:.3f}")
        ax = axes[row, col]
        ax.scatter(moons_ref[:, 0], moons_ref[:, 1], s=1, color="lightgray")
        ax.scatter(xs[:, 0], xs[:, 1], s=1, color=f"C{row}")
        ax.set_title(f"{name}, {n} steps\nED {ed:.3f} | precision {prec:.2f}", fontsize=10)
        ax.set_xlim(-6, 6)
        ax.set_ylim(-4, 4)
        ax.set_aspect("equal")
fig.suptitle("Both improve with more sampling steps; with few steps both fail (one seed only)")
plt.tight_layout()
plt.show()

读这张图要克制：这里**只有一个种子**。两种方法都随步数变好，2 步时都失败；中等步数（5–20 步）时 FM 的欧拉采样通常更好，100 步时两者都接近噪声地板。这是不是稳定的规律、差距是否大于种子间的波动——正是作业研究实验 4 要用 3 个种子回答的问题。

### ε、数据、速度：三种预测可以互相换算

在**同一个状态** $x_t$ 上，知道了网络的一种预测，就能解出另外两种——因为 $x_t$ 本身就是数据与噪声按已知系数的混合。记 $a, s$ 为数据、噪声的系数（FM：$a = t, s = 1 - t$；DDPM：$a = \sqrt{\bar\alpha}, s = \sqrt{1 - \bar\alpha}$）：

| 已知网络输出 | 求干净数据 $\hat x$ | 求噪声 $\hat\varepsilon$ | 求 FM 速度 $\hat u = \hat x - \hat\varepsilon$ |
| --- | --- | --- | --- |
| FM 速度 $\hat u$（FM 约定，$t$） | $x_t + (1 - t)\,\hat u$ | $x_t - t\,\hat u$ | — |
| 噪声 $\hat\varepsilon$（FM 约定，$t$） | $(x_t - (1 - t)\,\hat\varepsilon)/t$ | — | $(x_t - \hat\varepsilon)/t$ |
| 噪声 $\hat\varepsilon$（DDPM 约定，$\bar\alpha$） | $(x_t - \sqrt{1-\bar\alpha}\,\hat\varepsilon)/\sqrt{\bar\alpha}$ | — | 先用 `ddpm_to_fm` 换到 FM 状态，再用上一行 |
| $\hat v$（Salimans & Ho 2022，DDPM 约定） | $\sqrt{\bar\alpha}\,x_t - \sqrt{1-\bar\alpha}\,\hat v$ | $\sqrt{1-\bar\alpha}\,x_t + \sqrt{\bar\alpha}\,\hat v$ | 同上 |

两个提醒：① Salimans & Ho 的 v-prediction $v = \sqrt{\bar\alpha}\,\varepsilon - \sqrt{1-\bar\alpha}\,x_{\text{data}}$ **不是** FM 的速度（它在 DDPM 约定下，方向是"数据 → 噪声"）；② 从 $\hat\varepsilon$ 求数据或速度要**除以 $t$（或 $\sqrt{\bar\alpha}$）**，在接近纯噪声的一端会把网络的误差放大很多倍——Salimans & Ho §4 正是用这一点论证"少步采样时 ε 参数化吃亏"。

工具包的 `generative.convert_prediction(pred, x_t, 原参数化, 目标参数化, t_fm=... 或 alpha_bar=...)` 实现了这张表。用真值验证每一对换算：

In [ ]:
torch.manual_seed(0)
xd, ep = torch.randn(1000, 2) * 2 + 1, torch.randn(1000, 2)
t = torch.rand(1000, 1) * 0.98 + 0.01
ab = generative.fm_to_ddpm_time(t)                       # 与 t 信噪比相同的 ᾱ
x_fm = t * xd + (1 - t) * ep
x_vp = ab.sqrt() * xd + (1 - ab).sqrt() * ep
truth_fm = {"eps": ep, "data": xd, "velocity": xd - ep}
truth_vp = {"eps": ep, "data": xd, "v": ab.sqrt() * ep - (1 - ab).sqrt() * xd}
for label, truth, x, kw in (("FM 约定（t_fm=）", truth_fm, x_fm, {"t_fm": t}),
                            ("DDPM 约定（alpha_bar=）", truth_vp, x_vp, {"alpha_bar": ab})):
    print(f"{label}：由行换算到列，与真值的最大误差")
    print(" " * 10 + "".join(f"{b:>10s}" for b in truth))
    for a in truth:
        errs = [(generative.convert_prediction(truth[a], x, a, b, **kw) - truth[b]).abs().max().item() for b in truth]
        print(f"{a:>10s}" + "".join(f"{e:10.1e}" for e in errs))

### 同一个模型，两种语言

换算是精确的，所以可以做一件有趣的事：把 **FM 训练的速度模型**放进 **DDPM 的祖先采样器**。采样器每一步给出 DDPM 状态 $x$ 与 $\tau$；我们 ① 把状态和时间换到 FM 约定，② 问 FM 模型要速度，③ 把速度换算成噪声 $\hat\varepsilon$ 交回去（同一对 $(x_{\text{data}}, \varepsilon)$ 在两种约定下是同一个东西，所以 $\hat\varepsilon$ 可以直接用）：

In [ ]:
def fm_as_eps_model(fm_model):
    """把 FM 训练的速度模型包装成 DDPM 采样器需要的 eps_model(x_ddpm, τ)"""
    def eps_model(x_ddpm, tau):
        ab = alpha_bar(torch.tensor(tau))
        # 在这里写你的代码
        raise NotImplementedError
    return eps_model


for n in (5, 20, 100):
    native = ddpm_sample(ddpm_moons, noise((2000, 2)), n)
    borrowed = ddpm_sample(fm_as_eps_model(fm_moons), noise((2000, 2)), n)
    print(f"{n:3d} 步 DDPM 采样器：DDPM 训练的模型 ED {generative.energy_distance(native, moons_ref):.4f}"
          f" | FM 训练的模型 ED {generative.energy_distance(borrowed, moons_ref):.4f}")

FM 训练的模型在 DDPM 采样器里工作得和原生 DDPM 模型差不多一样好（少步时甚至更好）——这是换算函数正确的端到端证据，也是"**DDPM 可以看作 Flow Matching 框架下的一种特定选择**"的具体含义：选了另一条高斯路径（方差保持的 VP 路径）、另一个预测目标（ε），外加一个随机采样器。它们学的是同一类对象。

但**"可以互相换算"不等于"换算后用哪个采样器都一样好"**：反过来把 DDPM 的 ε 模型换算成速度、交给均匀步长的欧拉法，少步时会差很多（调研实测：20 步 ED 0.33 vs 原生 0.017），原因就是上面的"除以 $t$"——DDPM 的起点在 FM 约定下是 $t \approx 0.0065$，误差被放大约 150 倍。

### 同一个 U-Net，把 DDPM 搬到 MNIST

第 16 周作业的 MNIST 训练循环，同样只改两行。为了在 CPU 上几分钟内跑完，这里只训练 800 步：

In [ ]:
x_train, y_train = generative.load_mnist(size=14, train=True, n=10000, seed=0)
experiments.set_seed(0)
ddpm_unet = generative.SmallUNet(img_size=14).to(device)
optimizer = torch.optim.AdamW(ddpm_unet.parameters(), lr=2e-3)
STEPS_DDPM = 800
losses = []
t0 = time.time()
for step in range(STEPS_DDPM):
    idx = torch.randint(0, len(x_train), (128,))
    x_data = x_train[idx].to(device)
    eps = torch.randn_like(x_data)
    tau = torch.rand(len(x_data), 1, 1, 1, device=device)
    # 在这里写你的代码
    raise NotImplementedError
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f"MNIST DDPM {STEPS_DDPM} 步用时 {time.time() - t0:.0f} 秒 | 损失 {np.mean(losses[:50]):.3f} → {np.mean(losses[-50:]):.3f}")
ddpm_unet.eval()
mnist_ddpm = ddpm_sample(ddpm_unet, noise((64, 1, 14, 14), seed=1).to(device), 100).cpu()
show_grid(mnist_ddpm, title=f"DDPM on MNIST 14x14 ({STEPS_DDPM} training steps, 100 sampling steps)")
plt.show()

800 步的训练只够学出"笔画"，大部分还不像数字——和第 16 周作业里 FM 在 1500 步时的样子差不多。在 CPU 上把无条件生成训好需要长得多的训练（见作业的 Colab 配置）。但下一节会看到：**告诉网络"要哪个数字"之后，同样的训练预算就够生成清晰的数字了。**

---
## S3 · 条件生成与 classifier-free guidance

### 类别嵌入 + null 类别

想"指定生成数字 7"，就把类别 $y$ 也交给网络：$u_\theta(x, t, y)$。做法和处理时间一样——把类别经过一个 `nn.Embedding` 变成向量，拼进（或加到）网络的输入里。

Classifier-free guidance（Ho & Salimans 2022，Algorithm 1）的训练只多一行：**以概率 $p_{\text{uncond}}$ 把类别换成一个特殊的 null 标签 ∅**（原文 §4.2 在 64×64 ImageNet 上比较了 $p_{\text{uncond}} = 0.1$、0.2、0.5：0.1 与 0.2 差不多好，0.5 在整条 IS/FID 曲线上都更差，见 Table 1）。这样一个网络同时学会了两件事：条件预测 $u_\theta(x, t, y)$ 和无条件预测 $u_\theta(x, t, \varnothing)$。我们把 null 实现成"第 $K+1$ 个类别"（编号 $K$，工具包 `SmallUNet` 里是 `model.null_class`）。（TeaPearce 的实现把被丢弃的标签变成全零、保留的标签变成**负的** one-hot——训练和采样一致所以能用，但不直观，我们不照搬。）

### 引导：沿"更像这一类"的方向多走一点

采样时把两个预测组合起来（Ho & Salimans eq. (6)，原文写的是 ε，对速度一样适用）：

$$\tilde u = (1 + w)\, u_\theta(x, t, y) - w\, u_\theta(x, t, \varnothing) = u_\theta(x, t, y) + w\,\big[u_\theta(x, t, y) - u_\theta(x, t, \varnothing)\big]$$

方括号是"有条件比没条件多出来的那部分"——指向"更像类别 $y$"的方向；$w$ 越大，沿这个方向推得越远。**$w = 0$ 就是普通的条件模型**（不加引导）。

> ⚠️ **$w$ 有两套约定**：本课程统一用 Ho & Salimans 原文的约定（上式）。MIT 讲义（§5.2 eq. (65)）与 MIT Lab 写成 $\tilde u = (1 - w_{\text{MIT}})\,u_\varnothing + w_{\text{MIT}}\,u_y$，那里 $w_{\text{MIT}} = 1$ 才是不加引导。两者的关系是 $w_{\text{MIT}} = 1 + w_{\text{Ho}}$——MIT Lab 里的 guidance 3 相当于我们的 $w = 2$。

因为速度与 ε 之间的换算对网络输出是线性的（系数只依赖 $x_t$ 和 $t$），在 ε 上做引导和在速度上做引导是同一件事。

### 先在 2D 上看清楚：三个重叠的高斯

三个类别的高斯中心排成一个半径 1.5 的三角形，标准差 0.8，**刻意让它们重叠**——这样"条件模型生成的点被判成目标类的比例"不会是 100%：就连真实数据，最好的（贝叶斯最优）分类器也只能分对约 91%。

In [ ]:
K, NULL = 3, 3                       # 3 个类别；编号 3 的嵌入是 null（"不告诉你类别"）
MEANS = generative.gaussian_mixture_means(K, radius=1.5)


def sample_3g(n):
    return generative.sample_gaussian_mixture(n, k=K, radius=1.5, std=0.8, return_labels=True)


def nearest_mean(x):
    # 等权、等方差的高斯混合：离哪个中心最近就判成哪一类，这就是贝叶斯最优分类器
    return torch.cdist(x, MEANS).argmin(dim=1)


class CondVelocityMLP(nn.Module):
    def __init__(self, hidden=128, emb_dim=16):
        super().__init__()
        self.emb = nn.Embedding(K + 1, emb_dim)          # K 个类别 + 1 个 null
        self.net = nn.Sequential(nn.Linear(3 + emb_dim, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 2))

    def forward(self, x, t, y):
        t = torch.as_tensor(t, dtype=x.dtype).reshape(-1, 1).expand(len(x), 1)
        # 在这里写你的代码
        raise NotImplementedError


def cfg_combine(pred_cond, pred_uncond, w):
    # Ho & Salimans 2022 eq. (6)：w = 0 就是普通的条件模型
    # 在这里写你的代码
    raise NotImplementedError


def guided(model, y, w, null):
    """把条件模型包装成欧拉法需要的 u(x, t)：y 是每个样本的目标类别"""
    y_null = torch.full_like(y, null)

    def u(x, t):
        return cfg_combine(model(x, t, y), model(x, t, y_null), w)
    return u


ck.cfg_combine(cfg_combine)
x_check, y_check = sample_3g(20000)
print(f"贝叶斯最优分类器在真实数据上的准确率: {(nearest_mean(x_check) == y_check).float().mean():.3f}")

In [ ]:
experiments.set_seed(0)
cond_mlp = CondVelocityMLP()
optimizer = torch.optim.Adam(cond_mlp.parameters(), lr=1e-3)
P_UNCOND = 0.2
t0 = time.time()
for step in range(5000):
    x_data, y = sample_3g(512)
    # 在这里写你的代码
    raise NotImplementedError
    eps = torch.randn_like(x_data)
    t = torch.rand(512, 1)
    xt = t * x_data + (1 - t) * eps
    loss = ((cond_mlp(xt, t, y) - (x_data - eps)) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"条件 MLP 5000 步用时 {time.time() - t0:.1f} 秒")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for c in range(K):
    axes[0].scatter(x_check[:3000][y_check[:3000] == c, 0], x_check[:3000][y_check[:3000] == c, 1], s=1)
axes[0].set_title("Real data: three overlapping classes")
for ax, w in zip(axes[1:], (0, 1, 3)):
    fracs, spreads = [], []
    for c in range(K):
        y_target = torch.full((1000,), c)
        xs = fm_sample(guided(cond_mlp, y_target, w, NULL), noise((1000, 2), seed=c), 100)
        fracs.append((nearest_mean(xs) == c).float().mean().item())
        spreads.append(xs.std(dim=0).mean().item())
        ax.scatter(xs[:, 0], xs[:, 1], s=1)
    print(f"w = {w}：判为目标类的比例 {np.mean(fracs):.3f} | 类内标准差 {np.mean(spreads):.3f}（真值 0.8）")
    ax.set_title(f"w = {w}: on-target {np.mean(fracs):.2f}, spread {np.mean(spreads):.2f}")
for ax in axes:
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
fig.suptitle("Stronger guidance pushes classes apart and shrinks them: more on-target, less diverse")
plt.tight_layout()
plt.show()

$w = 0$ 时"判为目标类的比例"约等于真实数据上的贝叶斯准确率——条件模型学对了：它忠实地生成了**包括重叠区在内**的条件分布。引导越强，各类越往远离彼此的方向挤、越收缩：更"像"目标类，但更不多样，**也离真实的条件分布越来越远**。这与 Ho & Salimans 原文的 2D 示意图（Figure 2）一致。

### MNIST：指定生成某个数字

同样的配方用在 MNIST 14×14 上：`SmallUNet(num_classes=10)` 内置了 11 个类别嵌入（编号 10 是 null），$p_{\text{uncond}} = 0.1$，FM 训练 1000 步：

In [ ]:
experiments.set_seed(0)
cond_unet = generative.SmallUNet(img_size=14, num_classes=10).to(device)
optimizer = torch.optim.AdamW(cond_unet.parameters(), lr=2e-3)
STEPS_COND, P_UNCOND_MNIST = 1000, 0.1
print("条件 SmallUNet 参数量:", generative.count_parameters(cond_unet), "| null 类别编号:", cond_unet.null_class)
t0 = time.time()
for step in range(STEPS_COND):
    idx = torch.randint(0, len(x_train), (128,))
    x_data, y = x_train[idx].to(device), y_train[idx].to(device)
    drop = torch.rand(len(y), device=device) < P_UNCOND_MNIST
    y = torch.where(drop, torch.full_like(y, cond_unet.null_class), y)        # 以概率 0.1 丢掉类别
    eps = torch.randn_like(x_data)
    t = torch.rand(len(x_data), 1, 1, 1, device=device)
    xt = t * x_data + (1 - t) * eps
    loss = ((cond_unet(xt, t, y) - (x_data - eps)) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
cond_unet.eval()
print(f"条件 MNIST FM {STEPS_COND} 步用时 {time.time() - t0:.0f} 秒")

**怎么判断生成的数字"对不对"？** 用一个在真实数字上训练的分类器去认（工具包 `generative.train_mnist_classifier`，第 10 周的小 CNN 结构，几秒训完）。每个数字生成 20 张，报告：被判成目标数字的比例、预测类别直方图的归一化熵（1 = 10 个数字均匀出现）、以及**多样性**——同一目标数字的 20 张图两两之间像素距离的平均。

In [ ]:
clf = generative.train_mnist_classifier(size=14)
targets = torch.arange(10).repeat_interleave(20)              # 每个数字 20 张
rows, grids = {}, {}
for w in (0, 1, 3):
    t0 = time.time()
    xs = fm_sample(guided(cond_unet, targets.to(device), w, cond_unet.null_class),
                   noise((200, 1, 14, 14), seed=1).to(device), 50).cpu()
    s = generative.classifier_scores(clf, xs, targets)
    rows[f"w = {w}"] = dict(s, diversity=generative.within_class_diversity(xs, targets))
    grids[w] = xs.view(10, 20, 1, 14, 14)[:, :10].reshape(100, 1, 14, 14)     # 每个数字取前 10 张
    print(f"w = {w}：判为目标数字 {s['frac_target']:.3f} | 平均置信度 {s['mean_confidence']:.3f} | "
          f"类别熵 {s['class_entropy']:.3f} | 多样性 {rows[f'w = {w}']['diversity']:.2f} | 采样 {time.time() - t0:.1f} 秒")

fig, axes = plt.subplots(1, 3, figsize=(15, 5.4))
for ax, w in zip(axes, (0, 1, 3)):
    show_grid(grids[w], nrow=10, title=f"w = {w}", ax=ax)
fig.suptitle("Rows are the requested digits 0-9: guidance makes each row on-target and more alike")
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

### 阴性对照：分类器很有把握，就说明样本好吗？

把**纯高斯噪声**——也就是上面采样器的起点——直接交给分类器：

In [ ]:
pure_noise = noise((200, 1, 14, 14), seed=1)                   # 恰好是上面每次采样的起点
x_real, y_real = generative.load_mnist(size=14, train=False, n=200, seed=3)
rows["pure noise"] = dict(generative.classifier_scores(clf, pure_noise, targets),
                          diversity=generative.within_class_diversity(pure_noise, targets))
rows["real digits"] = dict(generative.classifier_scores(clf, x_real, y_real),
                           diversity=generative.within_class_diversity(x_real, y_real))
print(f"{'':12s} {'判为目标':>8s} {'平均置信度':>8s} {'置信>0.9':>8s} {'类别熵':>8s} {'多样性':>8s}")
for name in ("real digits", "pure noise", "w = 0", "w = 1", "w = 3"):
    r = rows[name]
    print(f"{name:12s} {r['frac_target']:8.3f} {r['mean_confidence']:8.3f} {r['frac_confident']:8.3f} "
          f"{r['class_entropy']:8.3f} {r['diversity']:8.2f}")
counts = rows["pure noise"]["pred_counts"]
top = int(np.argmax(counts))
frac_top = max(counts) / len(pure_noise)
print(f"\n纯噪声的预测分布: {counts} —— {frac_top:.0%} 被判成了 {top}")
print(f"如果你要的数字恰好是 {top}：纯噪声的'命中率'就是 {frac_top:.0%}")
ck.noise_control(frac_top, rows["pure noise"]["class_entropy"])

分类器对纯噪声也会给出确定的答案——大部分被判成同一个数字，平均置信度也不低。所以：

- 只报"判为目标数字的比例"是不够的：如果目标恰好是那个数字，**纯噪声就能刷出很高的命中率**；
- 必须同时报告**类别熵**（纯噪声 ≪ 1，真实数字 ≈ 1）、**多样性**，并把**纯噪声阴性对照**作为表格里的一行。这和第 9 周"输入全零的基线"是同一种思路：先知道"什么都没学到"长什么样。

回到权衡：引导把"判为目标"的比例推到接近 1，同时多样性单调下降、逐渐低于真实数据。原文在 ImageNet 上看到同样的权衡（Table 1 是 64×64、Table 2 是 128×128，$w$ 从 0 扫到 4）：最好的 FID 出现在**很小**的引导——64×64 上是 $w = 0.1$，128×128 上是 $w = 0.3$；最好的 Inception Score 出现在强引导（$w \ge 4$，即扫描范围的上端）。"选多大的 $w$"取决于你更在乎保真还是多样。

### 简介：在潜空间里生成（Stable Diffusion 的思路）

在 512×512 的彩色图上直接做扩散，每一步都要处理几十万个像素，代价极高。Rombach 等人（*High-Resolution Image Synthesis with Latent Diffusion Models*, CVPR 2022, [arXiv:2112.10752](https://arxiv.org/abs/2112.10752)）的做法是分两段：

1. 先训练一个**自编码器**（与第 15 周的 VAE 同一类模型），把图像压到一个小得多的**潜空间**（例如 64×64×4），解码器能把潜向量还原成图像；
2. 在**潜空间里**训练扩散 / Flow Matching 模型——它生成的是潜向量，最后交给解码器变成图像。

文字条件通过交叉注意力（第 11 周）注入网络，采样时同样使用 classifier-free guidance。概念上它就是今天的全部内容 + 第 15 周的 VAE：**"在哪个空间里做生成"本身也是一个可以选择的设计**。MIT 6.S184 2026 版 Lab 3 有完整的 VAE + 潜空间扩散参考实现（需要 GPU）。

---
## S4 · Capstone 工作时段

本周进入主实验推进期（第 18 周要拿出主结果，≥ 3 个种子）。动手之前，先对照下面的清单逐项自查：

- [ ] 主实验的**自变量**只有一个吗？其余条件是否像今天的 FM vs DDPM 一样"只改你想研究的那两行"？
- [ ] 冗余超参数（例如学习率）是否为每个条件单独调过（第 9 周 Tuning Playbook）？
- [ ] 每个条件 ≥ 3 个种子；时间预算够吗（单次耗时 × 种子数 × 条件数）？
- [ ] 主指标之外，有没有一个**互补的指标**或一个**阴性对照**，能防止你被一个好看的数字骗了（今天的能量距离与"纯噪声被判为 8"）？
- [ ] 实验记录本里，本周的每一次运行都有记录吗？

### 本周小结

- DDPM 前向：$x_t = \sqrt{\bar\alpha_t}\,x_{\text{data}} + \sqrt{1-\bar\alpha_t}\,\varepsilon$——另一条高斯插值；**DDPM 的 $t = 0$ 是数据，FM 的 $t = 0$ 是噪声**；换算按信噪比对齐，不是 $1 - t$
- 从 FM 到 DDPM：训练只改两行（路径、目标），采样换成祖先采样（每步去噪 + 加回新噪声，随机的）
- ε、数据、速度（以及 v）在同一状态上可以互相换算；FM 训练的模型可以放进 DDPM 采样器——DDPM 是同一框架下的一种选择。但换算后的采样器未必一样好（从 ε 求速度要除以 $t$）
- 条件生成：类别嵌入 + null 类别，训练时以 $p_{\text{uncond}}$ 丢掉类别；CFG：$(1+w)\,u_y - w\,u_\varnothing$（Ho 约定，$w = 0$ 不引导；$w_{\text{MIT}} = 1 + w_{\text{Ho}}$）
- 引导越强，越"像目标类"、越不多样；分类器对纯噪声也很有把握——报告类别熵、多样性与阴性对照
- 潜空间扩散：先压缩（VAE），再在潜空间里生成